DH演習2026 · 教材 第44版 · 2026-r44 · 2026-10-29

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第44版
教材ID: 2026-r44
教材日付: 2026-10-29
活動: place_identity
環境: Colab / Jupyter
ソース: e3d79cdb5300ec1d0937c1c5211b0882cbcd4e7c
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第8回：同じ地名は、同じ場所を指すか

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

資料：[新板大坂之図](https://image.oml.city.osaka.lg.jp/da/detail?tilcod=0000000021-OSK0186760)と[難波橋の絵葉書](https://image.oml.city.osaka.lg.jp/da/detail?tilcod=0000000021-OSK0157268)（大阪市立図書館、CC0）、[Wikidata](https://www.wikidata.org/)（CC0）

## 1. 古地図で橋名を探す

N1は1657年の原図を1980年に複製した地図で、東が上（北が左）です。ノートブックは原画像（約25 MB）を取得し、川の部分と橋名の部分を拡大して表示します。

In [ ]:
from IPython.display import display
from PIL import Image

url = "https://image.oml.city.osaka.lg.jp/da/download/?id=0000000021-OSK0186760&size=org&type=image&file=%2F%E5%86%99%E7%9C%9F%E3%83%BB%E7%B5%B5%E3%81%AF%E3%81%8C%E3%81%8D%E3%83%BB%E5%8F%A4%E6%96%87%E6%9B%B8%E3%83%BB%E5%9C%B0%E5%9B%B3%2Fr1002001.jpg"
!curl -fso N1.jpg "{url}"
n1 = Image.open("N1.jpg")
display(n1.crop((100, 2300, 2000, 3600)).reduce(2), n1.crop((950, 2670, 1250, 2920)))

## 2. 絵葉書の年と景観を確かめる

N2の目録の刊年は[1915]です。角括弧は目録が補った年を示し、撮影日とは限りません。ライオン像・路面電車・欄干を探してください。

In [ ]:
url = "https://image.oml.city.osaka.lg.jp/da/download/?id=0000000021-OSK0157268&size=view&type=image&file=%2F%E5%86%99%E7%9C%9F%E3%83%BB%E7%B5%B5%E3%81%AF%E3%81%8C%E3%81%8D%E3%83%BB%E5%8F%A4%E6%96%87%E6%9B%B8%E3%83%BB%E5%9C%B0%E5%9B%B3%2Fc0150001.jpg"
!curl -fso N2.jpg "{url}"
Image.open("N2.jpg")

## 3. Wikidataで候補を検索する

`query`を変えてから、次の取得セルを実行してください。ノートブックは検索語をWikidataへ送り、候補の名前・説明・座標を表示します。座標は現在の代表点で、古地図に描かれた位置ではありません。

In [ ]:
query = "難波橋"

In [ ]:
#@title 操作：Wikidataから候補を取得して表示する
import pandas as pd
import requests

api = "https://www.wikidata.org/w/api.php"
headers = {"User-Agent": "dh-enshu-2026-colab/1.0"}
found = requests.get(api, headers=headers, params={"action": "wbsearchentities", "search": query, "language": "ja", "uselang": "ja", "type": "item", "limit": "10", "format": "json"}).json()["search"]
ids = "|".join(c["id"] for c in found)
entities = requests.get(api, headers=headers, params={"action": "wbgetentities", "ids": ids, "props": "claims", "format": "json"}).json()["entities"] if ids else {}
rows = []
for c in found:
    point = entities[c["id"]]["claims"].get("P625", [{}])[0].get("mainsnak", {}).get("datavalue", {}).get("value", {})
    rows.append({"id": c["id"], "名前": c.get("label", ""), "説明": c.get("description", ""), "緯度": point.get("latitude"), "経度": point.get("longitude")})
candidates = pd.DataFrame(rows)
candidates

## 4. 候補を絞る条件を変える

`exclude_stations`を`True`にすると、名前に「駅」を含む候補を除きます。絞り込みは同定ではありません。残った候補が資料の示す対象と合う理由を、資料に基づいて説明してください。

In [ ]:
exclude_stations = False

kept = candidates[~candidates["名前"].str.contains("駅")] if exclude_stations else candidates
print("全候補：", len(candidates), "絞り込み後：", len(kept))
kept

## 5. 判断を表に記録する

同じ読みでも別の対象を指し、違う名前でも同じ対象を指すことがあります。

| 表記 | 指しているもの | 表に残す関係 |
| --- | --- | --- |
| 難波橋 | 川を渡る橋 | 橋の名前 |
| ライオン橋 | 同じ橋の愛称 | 橋の別名 |
| なにわ橋駅 | 京阪中之島線の駅 | 橋とは別の対象 |

資料ごとに、`label_as_seen`（資料の表記）、`period_label`（その資料から言える時期）、`location_claim`（位置について言えること）、`evidence_and_reason`（根拠）を記入してください。`decision`は`accept`（採用）・`defer`（保留）・`reject`（却下）です。

In [ ]:
rows = pd.DataFrame([
    dict(record_id="N1", label_as_seen="", place_id="naniwabashi", period_label="", location_claim="", decision="defer", evidence_and_reason=""),
    dict(record_id="N2", label_as_seen="", place_id="naniwabashi", period_label="", location_claim="", decision="defer", evidence_and_reason=""),
])
rows

[大阪市の橋の解説](https://www.city.osaka.lg.jp/kensetsu/page/0000023795.html)は、以前の橋が現在の堺筋より一筋西にあったとしています。N1とN2が同じ橋を指すと判断しても、描かれた位置が同じかは別の問いです。座標を決めるために、次にどの資料が必要でしょうか。

## 確認問題

中之島公園の案内図にある「京阪 なにわ橋駅」を`station_row`に記録してください。橋とは別の対象として、`label_as_seen`に表記、`evidence_and_reason`に根拠を書いてください。

In [ ]:
station_row = dict(record_id="N4", label_as_seen="", place_id="naniwabashi", decision="defer", evidence_and_reason="")

In [ ]:
#@title 答えを確かめる
import unicodedata

if "なにわ橋駅" not in unicodedata.normalize("NFKC", station_row["label_as_seen"]):
    print("結果：不合格（`label_as_seen`に「なにわ橋駅」と書いてください）")
elif station_row["place_id"] == "naniwabashi" and station_row["decision"] != "reject":
    print("結果：不合格（駅を橋と同じ場所にしています。別の`place_id`にするか、`naniwabashi`との対応を`reject`にしてください）")
elif not station_row["evidence_and_reason"].strip():
    print("結果：不合格（`evidence_and_reason`に根拠を書いてください）")
else:
    print("結果：合格")